# Ähnlichkeitssuche in Python

Dasselbe wie die Live-Demo in Java, diesmal in Python: dasselbe Embedding-Modell `multilingual-e5-small`, dieselben 643 Chunks aus AstroPlanners Wissensbasis, die Suche als Schleife über alle Chunks. Die Vektoren der Chunks sind vorberechnet und liegen im Ordner `index/`. Hier wird nur der Suchtext eingebettet.

1. Modell und Chunks laden
2. Ein Text wird ein Vektor
3. Die Suche: eine Schleife über alle Chunks
4. Die Suchen aus der Live-Demo
5. Eine eigene Suche

**Pflicht** (15 min): Schritte 1 bis 5. Am schnellsten mit *Laufzeit → Alle ausführen*, dann von oben lesen.

**Zusatz**: Suchen ohne die Präfixe `query: ` und `passage: `.

## Pflicht

### 1. Modell und Chunks laden

Beim ersten Mal lädt das Modell rund 490 MB von Hugging Face. Die Chunks und ihre Vektoren kommen aus dem Ordner.

In [ ]:
# Nur in Colab: Hilfsdateien und Daten aus dem Repo holen. Lokal tut die Zelle nichts.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repo = "/content/ki-workshop-handson"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dprinz/ki-workshop-handson.git", repo], check=True)
    os.chdir(f"{repo}/suche")

In [1]:
import time

import numpy as np

from helfer import chunk_text, einbetten, lade_chunks, lade_index, lade_modell, zeige

start = time.perf_counter()
lade_modell()
chunks = lade_chunks()
vektoren = lade_index()  # eine Zeile pro Chunk, je 384 Zahlen
print(f"{len(chunks)} Chunks, Index {vektoren.shape}, geladen in {time.perf_counter() - start:.1f} s")

643 Chunks, Index (643, 384), geladen in 7.2 s


Ein Chunk ist ein Abschnitt eines Dokuments mit höchstens 250 Wörtern. Die erste Zeile nennt Dokument und Abschnitt. So hat das Modell den Chunk bekommen, mit dem Präfix `passage: ` davor:

In [2]:
chunk = next(c for c in chunks if c["abschnitt"] == "Astronomical twilight")
print("passage: " + chunk_text(chunk))

passage: Twilight – Astronomical twilight
Astronomical twilight occurs when the geometric center of the Sun is between 18° and 12° below the horizon. During astronomical twilight, the sky is dark enough to permit astronomical observation of point sources of light such as stars, except in regions with more intense skyglow due to light pollution, moonlight, auroras, and other sources of light. Some critical observations, such as of faint diffuse items such as nebulae and galaxies, may require observation beyond the limit of astronomical twilight. Theoretically, the faintest stars detectable by the naked eye (those of approximately the sixth magnitude) will become visible in the evening at astronomical dusk, and become invisible at astronomical dawn.


### 2. Ein Text wird ein Vektor

Eine Frage bekommt das Präfix `query: `. So wurde e5 trainiert: Fragen mit `query: `, Texte mit `passage: `.

`einbetten` steht in `helfer.py`: Tokenizer, Modell, dann der Mittelwert über die Vektoren aller Token (Mean Pooling), auf Länge 1 normiert. Wie in Java und AstroPlanner.

In [3]:
for text in ["query: Wann beginnt die astronomische Dämmerung?", "query: Dämmerung"]:
    vektor = einbetten(text)
    print(f"{text}\nDimension {len(vektor)}, Länge {np.linalg.norm(vektor):.3f}")
    print(np.round(vektor[:8], 4), "…\n")

query: Wann beginnt die astronomische Dämmerung?
Dimension 384, Länge 1.000
[ 0.1014 -0.0272 -0.0424 -0.0483  0.0687 -0.06   -0.0084  0.0564] …

query: Dämmerung
Dimension 384, Länge 1.000
[ 0.0505 -0.0321 -0.0474 -0.1057  0.0572 -0.0415  0.0274  0.04  ] …



Eine ganze Frage und ein einzelnes Wort ergeben gleich viele Zahlen: 384. Die einzelnen Zahlen haben keine lesbare Bedeutung.

Java zeigte für die erste Frage `[0.0949, -0.0175, -0.0422, -0.0464, 0.0714, …]`. Die Zahlen hier weichen etwas ab: Java lädt eine quantisierte Fassung des Modells (ONNX, 118 MB), hier läuft das Original (470 MB).

### 3. Die Suche: eine Schleife über alle Chunks

Dieselbe Schleife wie auf der Folie in Java. Die Vektoren haben Länge 1, das Skalarprodukt ist deshalb schon der Kosinus. Die Distanz ist 1 − Kosinus, kleiner ist näher.

In [4]:
def suche(frage, vektoren, thema=None, k=5):
    treffer = []
    for chunk, vektor in zip(chunks, vektoren):
        if thema is not None and chunk["thema"] != thema:
            continue
        kosinus = np.dot(frage, vektor)  # beide Vektoren haben Länge 1: Skalarprodukt = Kosinus
        treffer.append((1 - kosinus, chunk))  # Distanz wie in Java und pgvector: 0 = gleiche Richtung
    treffer.sort(key=lambda t: t[0])
    return treffer[:k]

### 4. Die Suchen aus der Live-Demo

In [5]:
start = time.perf_counter()
frage = einbetten("query: Wann beginnt die astronomische Dämmerung?")
mitte = time.perf_counter()
treffer = suche(frage, vektoren)
ende = time.perf_counter()

zeige(treffer)
print(f"Embedding {(mitte - start) * 1000:.0f} ms, Suche {(ende - mitte) * 1000:.1f} ms über {len(chunks)} Chunks")

1  0.173  Twilight – Astronomical twilight
          Astronomical twilight occurs when the geometric center of the Sun is b …
2  0.186  Twilight – Between day and night
          because after the equinoxes the Sun's declination is then less than 4° …
3  0.196  Twilight – Between day and night
          astronomical dusk in the entire year is 48º33'43". The maximum latitud …
4  0.200  Twilight – Civil twilight
          Civil twilight is the period of time for which the geometric center of …
5  0.203  Twilight – Lasting from one day to the next
          night. Nautical twilight: between about 54°34' and 60°34' north or sou …
Embedding 6 ms, Suche 0.5 ms über 643 Chunks


Der Korpus beantwortet die nächste Frage nicht. Liefert die Suche nichts oder fünf Treffer?

In [6]:
zeige(suche(einbetten("query: Wie alt ist das Universum?"), vektoren))

1  0.206  Planetary nebula – Introduction
          A planetary nebula is a type of emission nebula consisting of an expan …
2  0.207  Globular cluster – Consequences
          The blue stragglers appear on the H–R diagram as a series diverging fr …
3  0.208  Galaxy – Super-luminous spiral
          Recently, researchers described galaxies called super-luminous spirals …
4  0.210  Galaxy – Evolution
          Once stars begin to form, emit radiation, and in some cases explode, t …
5  0.210  Globular cluster – Composition
          Globular clusters are generally composed of hundreds of thousands of l …


Die beiden Suchtexte, die Marvin im ersten Live-Lauf von AstroPlanner geschrieben hat, mit dem Themenfilter `filters` (100 Chunks):

In [7]:
zeige(suche(einbetten("query: OIII Filter"), vektoren, thema="filters"))
print()
zeige(suche(einbetten("query: narrowband OIII filter oxygen emission"), vektoren, thema="filters"))

1  0.159  Astronomical filter – Ultra-high contrast filters
          Known commonly as UHC filters, these filters consist of things which a …
2  0.161  Astronomical filter – Moon filters
          Neutral density filters, also known in astronomy as Moon filters, are …
3  0.167  HOO, SHO and LRGB – Narrowband filters
          Emission nebulae shine in a few spectral lines. The most important for …
4  0.167  HOO, SHO and LRGB – HOO
          HOO uses two filters: Ha is mapped to red, and OIII is mapped to both …
5  0.168  Astronomical filter – Specific emission line filters
          Specific emission line (or lines) filters are used to isolate lines of …

1  0.095  HOO, SHO and LRGB – Narrowband filters
          Emission nebulae shine in a few spectral lines. The most important for …
2  0.100  Astronomical filter – Narrowband
          Narrow-band filters are astronomical filters which transmit only a nar …
3  0.122  Doubly ionized oxygen – Ion
          Its emission of forbidden lin

Bester Treffer im Vergleich mit der Live-Demo:

| Suchtext | Java | hier |
|---|---|---|
| Wann beginnt die astronomische Dämmerung? | Twilight – Astronomical twilight, 0.165 | Twilight – Astronomical twilight, 0.173 |
| Wie alt ist das Universum? | Galaxy – Super-luminous spiral, 0.203 | Planetary nebula – Introduction, 0.206 |
| OIII Filter | Astronomical filter – Ultra-high contrast filters, 0.156 | dasselbe, 0.159 |
| narrowband OIII filter oxygen emission | HOO, SHO and LRGB – Narrowband filters, 0.093 | dasselbe, 0.095 |

Die Distanzen liegen hier um 0,002 bis 0,008 höher, wegen der quantisierten Fassung in Java. Die Reihenfolge bleibt bei drei Suchen gleich. Beim Universum liegen die ersten fünf Treffer so dicht beieinander (0,206 bis 0,210), dass schon diese kleine Abweichung die Reihenfolge ändert; Galaxy – Super-luminous spiral steht hier auf Rang 3.

Die Suche liefert immer k Treffer, auch wenn keiner passt. Ob einer passt, muss die Anwendung entscheiden.

### 5. Eine eigene Suche

Ein eigener Suchtext, Deutsch oder Englisch. Ideen: eine Frage zur Astrofotografie, eine Frage aus eurem Alltag, die der Korpus nicht beantwortet, ein einzelnes Wort, ein Tippfehler. Themen für den Filter: `filters`, `imaging`, `equipment`, `targets`.

Worauf achten: Wie weit ist der beste Treffer weg, verglichen mit der Dämmerung und dem Universum?

In [8]:
mein_suchtext = "Welcher Filter hilft gegen Lichtverschmutzung in der Stadt?"
mein_thema = None  # oder z. B. "filters"

zeige(suche(einbetten("query: " + mein_suchtext), vektoren, thema=mein_thema))

1  0.185  Astronomical filter – Broadband
          The broadband, or light pollution reduction (LPR), filters are designe …
2  0.186  Light pollution – Over-illumination
          stray light and energy costs; "Daylight lighting" demanded by citizens …
3  0.193  Light pollution – Effect on astronomy
          by filtering out spectral lines commonly emitted by sodium- and mercur …
4  0.194  Light pollution – Improving lighting fixtures
          capita energy consumption for public lighting in Italy. The same legis …
5  0.194  Light pollution – Effect on astronomy
          Astronomy is very sensitive to light pollution. The night sky viewed f …


## Zusatz: ohne Präfixe

e5 erwartet `query: ` vor der Frage und `passage: ` vor dem Chunk. Was passiert, wenn man beide weglässt? Dafür liegt ein zweiter Index im Ordner: dieselben Chunks, ohne `passage: ` eingebettet.

Aufgabe: Die Zelle unten so ändern, dass die Frage ohne `query: ` eingebettet und im Index `vektoren_ohne` gesucht wird. Dann den besten Treffer und seine Distanz mit Schritt 4 vergleichen. Ändert sich mehr als die Distanz?

In [9]:
vektoren_ohne = lade_index(praefix=False)

suchtext = "Wann beginnt die astronomische Dämmerung?"
frage = einbetten("query: " + suchtext)  # ändern: ohne Präfix
zeige(suche(frage, vektoren))  # ändern: im Index ohne Präfix

1  0.173  Twilight – Astronomical twilight
          Astronomical twilight occurs when the geometric center of the Sun is b …
2  0.186  Twilight – Between day and night
          because after the equinoxes the Sun's declination is then less than 4° …
3  0.196  Twilight – Between day and night
          astronomical dusk in the entire year is 48º33'43". The maximum latitud …
4  0.200  Twilight – Civil twilight
          Civil twilight is the period of time for which the geometric center of …
5  0.203  Twilight – Lasting from one day to the next
          night. Nautical twilight: between about 54°34' and 60°34' north or sou …


In [10]:
# @title Lösung (aufklappen)
demo_suchen = [
    ("Wann beginnt die astronomische Dämmerung?", None),
    ("Wie alt ist das Universum?", None),
    ("OIII Filter", "filters"),
    ("narrowband OIII filter oxygen emission", "filters"),
]


def bester(frage, index, thema):
    distanz, chunk = suche(frage, index, thema=thema)[0]
    return f"{distanz:.3f} {chunk['dokument']} – {chunk['abschnitt']}"


for text, thema in demo_suchen:
    print(text)
    print("  beide mit Präfix:   ", bester(einbetten("query: " + text), vektoren, thema))
    print("  beide ohne Präfix:  ", bester(einbetten(text), vektoren_ohne, thema))
    print("  nur Frage ohne:     ", bester(einbetten(text), vektoren, thema))

Wann beginnt die astronomische Dämmerung?
  beide mit Präfix:    0.173 Twilight – Astronomical twilight
  beide ohne Präfix:   0.170 Twilight – Astronomical twilight
  nur Frage ohne:      0.164 Twilight – Astronomical twilight
Wie alt ist das Universum?
  beide mit Präfix:    0.206 Planetary nebula – Introduction
  beide ohne Präfix:   0.197 Globular cluster – Consequences
  nur Frage ohne:      0.206 Globular cluster – Consequences
OIII Filter
  beide mit Präfix:    0.159 Astronomical filter – Ultra-high contrast filters
  beide ohne Präfix:   0.159 Astronomical filter – Ultra-high contrast filters
  nur Frage ohne:      0.167 Astronomical filter – Ultra-high contrast filters
narrowband OIII filter oxygen emission
  beide mit Präfix:    0.095 HOO, SHO and LRGB – Narrowband filters
  beide ohne Präfix:   0.108 HOO, SHO and LRGB – Narrowband filters
  nur Frage ohne:      0.111 HOO, SHO and LRGB – Narrowband filters


Auf diesem Korpus ändern die Präfixe wenig: Bei den drei Suchen mit Antwort bleibt der beste Treffer derselbe Abschnitt, die Distanz verschiebt sich um bis zu 0,016. Beim Universum wechselt der beste Treffer, und ohne beide Präfixe sinkt die Distanz von 0,206 auf 0,197. Die Frage ohne Antwort sieht dann näher aus, und ein Schwellwert, der mit Präfixen eingestellt wurde, passt nicht mehr genau.

Die dritte Zeile ist der häufigere Fehler in der Praxis: Der Index wurde mit `passage: ` gebaut, die Frage vergisst `query: `. Wie gut die Suche dann noch ist, zeigt eine Messung über viele Fragen, nicht ein Beispiel. Das kommt morgen.